# 02 - Preprocesado

Aca armo el pipeline de datos que voy a reusar en todos los notebooks de entrenamiento.
Redimensiono las imagenes a 224x224, aplico augmentation en train, y calculo
los pesos de clase para compensar el desbalance del dataset.

In [ ]:
!pip install tensorflow-datasets gdown --quiet

In [ ]:
import os

WORK_PATH = '/content/plantvillage'
os.makedirs(WORK_PATH, exist_ok=True)

In [ ]:
# descargo los archivos generados en el notebook 01
import gdown
gdown.download_folder(
    'https://drive.google.com/drive/folders/1OCOyDSR9C3TCzLthsMxJCmy6wcjM3pxs',
    output=WORK_PATH,
    quiet=True
)

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import json

print('tensorflow:', tf.__version__)
print('gpu disponible:', tf.config.list_physical_devices('GPU'))

tf.random.set_seed(42)
np.random.seed(42)

## Parametros globales

Uso 224x224 porque es el tamano que esperan VGG16, ResNet50 e InceptionV3.
Este pipeline aplica para los notebooks de transfer learning (04, 05, 06).
El notebook 03 (baseline CNN) usa 64x64 porque esa red no tiene pesos preentrenados
y no necesita esa resolucion — reducirla hace el entrenamiento mucho mas rapido.

In [ ]:
with open(f'{WORK_PATH}/dataset_info.json', 'r') as f:
    ds_info = json.load(f)

NUM_CLASSES = ds_info['num_classes']
CLASS_NAMES = ds_info['class_names']

IMG_SIZE   = 224
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

print('clases:', NUM_CLASSES)
print('tamano imagen:', IMG_SIZE)
print('batch size:', BATCH_SIZE)

## Carga del dataset

Uso el mismo split y la misma semilla que en el notebook 01 para que los
conjuntos de train, val y test sean exactamente los mismos en todos los notebooks.

In [ ]:
(ds_train_raw, ds_val_raw, ds_test_raw), info = tfds.load(
    'plant_village',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    with_info=True,
    as_supervised=True,
    shuffle_files=True,
    read_config=tfds.ReadConfig(shuffle_seed=42)
)

print('splits cargados')

## Pipeline de preprocesado

Para train aplico augmentation: flip, rotacion, zoom, brillo y contraste.
Para val y test solo redimensiono y normalizo, sin augmentation.

Los labels los dejo como enteros (no one-hot) porque voy a usar
sparse_categorical_crossentropy y class_weight en el entrenamiento,
y esa combinacion requiere labels enteros.

In [ ]:
# capas de augmentation definidas una sola vez
# value_range=(0,1) porque las imagenes ya estan normalizadas antes de entrar aqui
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 1)),
    tf.keras.layers.RandomContrast(0.2),
], name='augmentation')

def preprocess_train(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    # expand_dims porque las capas de keras esperan dimension de batch
    image = augmentation(tf.expand_dims(image, 0), training=True)[0]
    return image, label

def preprocess_val(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return image, label

In [ ]:
ds_train = (
    ds_train_raw
    .map(preprocess_train, num_parallel_calls=AUTOTUNE)
    .shuffle(buffer_size=2000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

ds_val = (
    ds_val_raw
    .map(preprocess_val, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

ds_test = (
    ds_test_raw
    .map(preprocess_val, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# verifico que las shapes sean las esperadas
for imagenes, labels in ds_train.take(1):
    print('shape imagenes:', imagenes.shape)
    print('shape labels:  ', labels.shape)
    print('dtype labels:  ', labels.dtype)

## Por que uso pesos de clase?

El dataset no esta balanceado, hay clases con mas de 5000 imagenes y otras con menos de 300.
Sin hacer nada al respecto, el modelo aprende a ignorar las clases pequeñas porque
equivocarse en ellas cuesta poco en el loss total.
Los pesos de clase le dicen al optimizador que los errores en clases minoritarias cuestan mas.

In [ ]:
# calculo el peso de cada clase a partir del conteo del split de train
conteo_train = ds_info['conteo_train']
total_train  = sum(conteo_train.values())

class_weight = {
    i: total_train / (NUM_CLASSES * conteo_train[name])
    for i, name in enumerate(CLASS_NAMES)
}

top5 = sorted(class_weight.items(), key=lambda x: x[1], reverse=True)[:5]
print('top 5 clases con mayor peso (las mas raras):')
for idx, peso in top5:
    print(f'  {CLASS_NAMES[idx]}: {peso:.3f}')

print()
bot5 = sorted(class_weight.items(), key=lambda x: x[1])[:5]
print('top 5 clases con menor peso (las mas frecuentes):')
for idx, peso in bot5:
    print(f'  {CLASS_NAMES[idx]}: {peso:.3f}')

In [ ]:
# las claves tienen que ser strings para que json.dump funcione
ruta_cw = f'{WORK_PATH}/class_weight.json'
with open(ruta_cw, 'w') as f:
    json.dump({str(k): v for k, v in class_weight.items()}, f, indent=2)

print('guardado en:', ruta_cw)

## Efecto del augmentation

Quiero ver visualmente que hace cada transformacion sobre una imagen real.
Asi confirmo que las transformaciones tienen sentido para este problema
(hojas de plantas vistas desde arriba, pueden estar en cualquier orientacion).

In [ ]:
for imagen_raw, _ in ds_train_raw.take(1):
    img = tf.cast(imagen_raw, tf.float32) / 255.0
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    fig, ejes = plt.subplots(2, 5, figsize=(15, 6))

    ejes[0][0].imshow(img.numpy())
    ejes[0][0].set_title('original')
    ejes[0][0].axis('off')

    for ax in ejes.flatten()[1:]:
        aug = augmentation(tf.expand_dims(img, 0), training=True)[0]
        ax.imshow(tf.clip_by_value(aug, 0, 1).numpy())
        ax.axis('off')

    plt.suptitle('efecto del augmentation sobre una imagen de entrenamiento', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{WORK_PATH}/augmentation_ejemplos.png', dpi=120)
    plt.show()